<a href="https://colab.research.google.com/github/Adri22K/ProjetoAndreaBD/blob/colab/Aula1_Pandas_CD_Trilha_A_alunos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 1 — Introdução ao Pandas e ao Pipeline de Ciência de Dados (CD)
### Trilha A — Chuva Intensa · Projeto Integrador

Nesta aula vamos aprender os principais comandos da biblioteca **pandas** fazendo a ingestão de um
arquivo CSV **sintético** que simula dados brutos de clima para São Paulo — o mesmo tipo de dado que,
na Aula 2, vamos aprender a obter diretamente da API do Open-Meteo.

O arquivo `dados_brutos_chuva_trilha_a.csv` foi construído **de propósito com problemas típicos** de
dados reais, para que possamos praticar os processos de limpeza exigidos pelo Pipeline de
**Ciência de Dados (CD)**:

| Etapa do Pipeline CD | O que vamos praticar nesta aula |
|---|---|
| Validação de esquema | `.info()`, `.dtypes`, `.isna()` |
| Limpeza | tratamento de nulos, duplicatas e outliers |
| Padronização | texto (nomes de cidade), tipos numéricos, datas |
| Dataset analítico | salvar o resultado tratado em `/data/processed` |

**Objetivo da aula:** ao final, vocês terão o próprio "kit de comandos" de pandas para reproduzir esse
processo de limpeza nos dados reais das Trilhas B (qualidade do ar) e C (produtividade agrícola).

## 0. Dicionário de dados (antes de começar)

| Coluna | Descrição | Tipo esperado |
|---|---|---|
| `data` | Data da observação | data (AAAA-MM-DD) |
| `cidade` | Cidade da estação meteorológica | texto |
| `temperatura_max_c` | Temperatura máxima do dia | número (°C) |
| `temperatura_min_c` | Temperatura mínima do dia | número (°C) |
| `umidade_pct` | Umidade relativa média do dia | número (0 a 100 %) |
| `pressao_hpa` | Pressão atmosférica média | número (hPa) |
| `vento_kmh` | Velocidade média do vento | número (km/h) |
| `precipitacao_mm` | Precipitação acumulada no dia | número (mm) |

**Atenção:** este dicionário é a nossa referência de "como os dados deveriam estar". Qualquer
divergência entre o que veremos no CSV e esta tabela é um problema de qualidade a ser tratado.

## 1. Ingestão do CSV

* Pandas: Biblioteca usada para manipular e analisar dados estruturados.
* Numpy: Biblioteca usada para cálculos numéricos e operações com arrays e matrizes.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)   #Exibe todas as colunas do DataFrame

Se estiver rodando no **Google Colab**, envie o arquivo `dados_brutos_chuva_trilha_a.csv` pela aba
de arquivos (ícone de pasta à esquerda) ou execute a célula abaixo antes de continuar.

In [ ]:
# Descomente as duas linhas abaixo se estiver no Google Colab e ainda não enviou o arquivo:
from google.colab import files
files.upload()

In [ ]:
#Lendo um arquivo do tipo csv e atribuindo a uma variável df
caminho = "dados_brutos_chuva_trilha_a.csv"
df = pd.read_csv(caminho)

A biblioteca Pandas disponibiliza diferentes recursos para carregar dados provenientes de diversos tipos de arquivos e fontes. As mais usadas:

* `read_csv()`: permite carregar dados armazenados em arquivos CSV, nos quais os valores são normalmente separados por vírgulas ou outros delimitadores. A função oferece parâmetros para definir o separador, identificar o cabeçalho, escolher a codificação do arquivo e especificar os tipos das colunas, entre outras configurações.

* `read_json()`: possibilita a leitura de dados estruturados no formato JSON (JavaScript Object Notation) e sua conversão para uma estrutura de dados do Pandas.

* `read_sql()`: permite obter dados armazenados em bancos de dados relacionais, como MySQL, PostgreSQL e SQL Server. A importação pode ser realizada por meio de uma consulta SQL ou pela leitura de uma tabela, utilizando uma conexão previamente configurada com o banco de dados.


In [ ]:
#Visualizando os dados
df.head(125)

In [ ]:
#Qual o tipo dos dados
type(df)

## 2. Primeiro olhar sobre os dados

Antes de limpar qualquer coisa, precisamos **entender** o que temos. Esses são os comandos que vocês
vão usar sempre, em qualquer dataset novo — não só nesta aula.

In [ ]:
# Quantas linhas e colunas?
df.shape

(746, 8)

(726, 8) é uma tupla

In [ ]:
# As 10 primeiras linhas
df.head(10)

,data,cidade,temperatura_max_c,temperatura_min_c,umidade_pct,pressao_hpa,vento_kmh,precipitacao_mm
0,2024-04-01,São Paulo,24.9,14.4,70.5,1022.6,2.7,0.0
1,2024-04-02,São Paulo,20.8,15.0,63.5,1008.8,12.5,0.0
2,2024-01-11,São Paulo,23.1,7.6,"78,4",1005.8,15.5,0.0
3,2023-09-02,São Paulo,"32,8",16.5,60.4,1014.1,0.8,17.9
4,2023-07-12,São Paulo,29.1,19.0,58.6,1010.5,6.9,67.9
5,2024-12-23,São Paulo,13.4,9.6,68.8,1020.8,3.5,0.0
6,2024-11-24,São Paulo,15.2,12.1,80.0,1015.0,9.8,0.0
7,2023-12-25,São Paulo,16.5,9.8,66.7,1012.6,5.8,0.0
8,2024-04-22,São Paulo,"22,2",19.7,105,1015.8,23.0,21.0
9,2023-07-06,São Paulo,30.3,20.9,59.1,1012.7,15.6,0.0


In [ ]:
# As últimas linhas
df.tail()

,data,cidade,temperatura_max_c,temperatura_min_c,umidade_pct,pressao_hpa,vento_kmh,precipitacao_mm
741,2024-08-02,São Paulo,"30,5",17.7,64.4,1012.0,40.0,0.0
742,2024-05-17,São Paulo,25.5,17.8,60.4,NaN,14.1,2.0
743,2024-06-21,São Paulo,32.0,"20,7",105,1008.3,7.7,0.0
744,2023-07-16,São Paulo,"31,4","20,7",64.5,1012.8,5.7,22.4
745,2023-06-25,São Paulo,30.0,19.3,66.5,1009.5,420.0,82.9


In [ ]:
# Nomes das colunas
df.columns

Index(['data', 'cidade', 'temperatura_max_c', 'temperatura_min_c',
       'umidade_pct', 'pressao_hpa', 'vento_kmh', 'precipitacao_mm'],
      dtype='object')

In [ ]:
# Tipo de dado (dtype) de cada coluna — primeiro sinal de problema!
df.dtypes

,0
data,object
cidade,object
temperatura_max_c,object
temperatura_min_c,object
umidade_pct,object
pressao_hpa,float64
vento_kmh,float64
precipitacao_mm,float64


In [ ]:
# Resumo estrutural: tipos, contagem de não-nulos e uso de memória
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 746 entries, 0 to 745
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   data               743 non-null    object 
 1   cidade             746 non-null    object 
 2   temperatura_max_c  714 non-null    object 
 3   temperatura_min_c  713 non-null    object 
 4   umidade_pct        718 non-null    object 
 5   pressao_hpa        721 non-null    float64
 6   vento_kmh          706 non-null    float64
 7   precipitacao_mm    717 non-null    float64
dtypes: float64(3), object(5)
memory usage: 46.8+ KB


In [ ]:
# Vamos visualizar uma coluna do tipo object
# O comando retorna uma "series" uma estrutura simples do dataframe composta por indice e valores
df['cidade']

In [ ]:
# Visualizando as colunas do tipo float
# Estrutura no modo tabela
# ['pressao_hpa', 'vento_kmh'] --> lista em python com 2 valores
df[['pressao_hpa', 'vento_kmh']]

In [ ]:
# Estatísticas descritivas das colunas numéricas
df.describe()

**Pare e observe:** de acordo com o dicionário de dados, `temperatura_max_c`, `temperatura_min_c` e
`umidade_pct` deveriam ser numéricas — mas o `.dtypes` provavelmente mostrou `object` (texto) para
elas. Isso é um problema real de CD que vamos resolver na Seção 5. Além disso, repare quantas
colunas o `.describe()` conseguiu descrever: só as que o pandas já reconhece como número.

## 3. Selecionando e filtrando dados

Comandos básicos de seleção que vamos usar o tempo todo no restante do pipeline.

In [ ]:
# Selecionar uma única coluna (retorna uma Series)
df["precipitacao_mm"]

In [ ]:
# Selecionar várias colunas (retorna um DataFrame)
df[["data", "cidade", "precipitacao_mm"]]

In [ ]:
# loc: selecionar por rótulo (linhas e colunas)
df.loc[0:3, ["data", "cidade", "temperatura_max_c"]]

In [ ]:
# iloc: selecionar por posição numérica
df.iloc[0:5, 0:3]

In [ ]:
# Filtrar linhas com uma condição: dias sem nenhuma chuva registrada

#Todas as linhas
df["precipitacao_mm"]

In [ ]:
#Registros com e sem chuva
df["precipitacao_mm"] == 0

In [ ]:
#Todas as linnhas com valores de precipitacao == 0
df[df["precipitacao_mm"] == 0]

In [ ]:
#Total de linhas com precipitacao_mm == 0
#shape apresenta as dimensões (número de linhas, número de colunas)
#shape[0] seleciona apenas a quantidade de linhas
df[df["precipitacao_mm"] == 0].shape[0]

In [ ]:
#código alternativo
(df["precipitacao_mm"] == 0).sum()

In [ ]:
# Tentando filtrar por cidade == "São Paulo" — quantas linhas "sobram"?
df[df["cidade"] == "São Paulo"].shape[0]

O número de linhas do filtro acima provavelmente é **menor** que o total de `df`. Isso não significa
que faltam dados de outras cidades — significa que o texto da coluna `cidade` está **inconsistente**
(maiúsculas, espaços, acentuação). Vamos resolver isso já na próxima seção.

## 4. Valores ausentes

Etapa central da **validação de esquema** do Pipeline CD: descobrir onde há dados faltando e decidir
o que fazer com cada caso.

In [ ]:
# Quantos valores ausentes existem por coluna?
df.isna().sum()

In [ ]:
# Em percentual, para dar noção de gravidade
(df.isna().mean() * 100).round(2)

**Como decidir o que fazer com cada ausência?**

- Se a coluna é **essencial** para o problema (ex.: `data`, sem a qual não sabemos quando o dado
  ocorreu) — a linha sem esse valor normalmente deve ser **removida**.
- Remove as linhas nas quais o valor é **NaN** ou **None**
- Se a coluna é uma **variável numérica** que será usada como atributo para o modelo (ex.:
  `umidade_pct`), a decisão de imputar (preencher) fica melhor guardada para o Pipeline de **ML**,
  ajustada **apenas nos dados de treino** — imputar agora, sem essa separação, pode causar vazamento
  de dados mais adiante. Por ora, vamos apenas **documentar** essas ausências.
> Importante: uma string vazia ("") não é automaticamente considerada um valor ausente. Para remover devemos convertê a string vazia em NaN antes.
```
df["precipitacao_mm"] = df["precipitacao_mm"].replace("", np.nan)
df = df.dropna(subset=["precipitacao_mm"])
```



In [ ]:
#conferindo as dimensões do dataset
linhas_antes_remocao = df.shape[0]
linhas_antes_remocao

In [ ]:
# Remover linhas em que a própria "data" está ausente (essa, sim, tratamos já aqui)
# As outras colunas mesmo que tenham dados ausentes são mantidas
# É necessário reorganizar os índices da tabela (reset_index) e descarta o índice antigo criando um novo. (drop=True)
df = df.dropna(subset=["data"]).reset_index(drop=True)

In [ ]:
linhas_depois_remocao = df.shape[0]
linhas_depois_remocao

print("Linhas removidas: ", linhas_antes_remocao - linhas_depois_remocao)

## 5. Linhas duplicadas

Dados coletados por API ou exportados de planilhas frequentemente vêm com registros repetidos —
seja por reexecução da coleta, seja por erro de exportação.

In [ ]:
# Quantas linhas são exatamente duplicadas?
df.duplicated().sum()

In [ ]:
# Visualizar algumas duplicatas (keep=False mostra todas as ocorrências, não só a repetida)
# df.duplicated(keep=False) --> localiza dos registros duplicados
# filtra apenas os registros duplicados.
# sort_values("data") --> ordena por data
# head(10) --> mostra os 10 primeiros registros
df[df.duplicated(keep=False)].sort_values("data").head(10)

In [ ]:
# Remover duplicatas, mantendo a primeira ocorrência
df = df.drop_duplicates().reset_index(drop=True)
df.shape

## 6. Padronizando texto (a coluna `cidade`)

Vamos resolver agora o problema identificado na Seção 3.

In [ ]:
# Quantas variações diferentes existem?
df["cidade"].unique()

In [ ]:
# Remover espaços nas pontas e padronizar capitalização
# strip() --> Remove espaços em branco no inicio e no final do texto
# upper() --> Transforma todas as letras em maiúsculas
df["cidade"] = df["cidade"].str.strip().str.upper()

In [ ]:
# Vamos verificar como ficou
df["cidade"].unique()

In [ ]:
# Ainda podem existir grafias diferentes (ex.: com e sem acento) — padronizamos com um mapeamento
#Dicionário que estabelece a relação entre valor encontrado --> valor utilizado
mapa_cidades = {
    "SÃO PAULO": "São Paulo",
    "SAO PAULO": "São Paulo",
}

# replace() procura na coluna cidade os valores do dicionário para serem substituídos.
df["cidade"] = df["cidade"].replace(mapa_cidades)
df["cidade"].unique()

In [ ]:
# Conferindo: agora o filtro por cidade deve bater com o total de linhas
df[df["cidade"] == "São Paulo"].shape[0], df.shape[0]

## 7. Conversão de tipos: decimais com vírgula e datas

Esta é a correção mais importante da aula: sem ela, colunas numéricas continuam sendo tratadas como
texto, e nenhuma conta ou gráfico funciona corretamente sobre elas.

In [ ]:
# Relembrando o problema
df[["temperatura_max_c", "temperatura_min_c", "umidade_pct"]].dtypes

In [ ]:
#Criando uma lista com os 3 campos que queremos alterar o tipo de object para float
colunas_numericas_sujas = ["temperatura_max_c", "temperatura_min_c", "umidade_pct"]

# Percorrer a lista e para cada coluna alterar o tipo
for col in colunas_numericas_sujas:
    df[col] = pd.to_numeric(
        df[col]                              # seleção da coluna
        .astype(str)                         # garante que dá para usar .str (converet todos os valores para texto)
        .str.replace(",", ".", regex=False),  # decimal brasileiro -> decimal internacional. regex=False --> trata a vírgula como caractere e não expressão regular
        errors="coerce"                       # errors="coerce" --> o que não converter vira NaN
    )

df[colunas_numericas_sujas].dtypes

In [ ]:
# Convertendo a coluna de data de texto para o tipo datetime do pandas
df["data"] = pd.to_datetime(df["data"], format="%Y-%m-%d")
df["data"].dtype

In [ ]:
# Conferindo o resultado
df.dtypes

## 8. Outliers e valores fora do domínio válido

Nem todo valor numérico "bate certo" — precisamos aplicar **regras de domínio** conhecidas sobre o
próprio fenômeno físico que estamos medindo.

In [ ]:
# describe() de novo, agora com as colunas já numéricas
df.describe()

Olhando o `min` e o `max` de cada coluna, algo deveria chamar atenção:

- `umidade_pct` só pode variar entre **0 e 100**;
- `precipitacao_mm` não pode ser **negativa**;
- `vento_kmh` acima de ~150 km/h já seria um evento extremo raríssimo para este contexto.

In [ ]:
# Identificando as linhas fora do domínio válido
#Umidades inválidas são aquelas fora do invervalo. Valor = True
umidade_invalida = ~df["umidade_pct"].between(0, 100)  # ~ operador de negação
#Ventos acima de 150 são consideradas situações atipicas. Valor = True
vento_invalido = df["vento_kmh"] > 150
#Precipitação negativa é um valor inválido . Valor = True
chuva_invalida = df["precipitacao_mm"] < 0

#loc --> permite selecionar dados no formato [linhas, colunas]
df.loc[umidade_invalida | vento_invalido | chuva_invalida,  # as linhas selecionadas devem atender a operação lógica | (ou), se 1 valor True a linha é selecionada
       ["data", "umidade_pct", "vento_kmh", "precipitacao_mm"]]   # Dados exibidos

In [ ]:
umidade_invalida.sum(), vento_invalido.sum(), chuva_invalida.sum()

In [ ]:
# Tratando como dado ausente (NaN) em vez de simplesmente apagar a linha inteira —
# assim preservamos as outras colunas, que continuam válidas naquele dia
# O que estamos fazando? Os dados que são inválidos são substituídos por NaN
df.loc[umidade_invalida, "umidade_pct"] = np.nan
df.loc[vento_invalido, "vento_kmh"] = np.nan
df.loc[chuva_invalida, "precipitacao_mm"] = np.nan

df.describe()

In [ ]:
df.loc[umidade_invalida | vento_invalido | chuva_invalida,  # as linhas selecionadas devem atender a operação lógica | (ou), se 1 valor True a linha é selecionada
       ["data", "umidade_pct", "vento_kmh", "precipitacao_mm"]]   # Dados exibidos

In [ ]:
# Veja que a quantidade não foi alterada
umidade_invalida.sum(), vento_invalido.sum(), chuva_invalida.sum()

## 9. Ordenação e checagem da linha do tempo

Como este é um dado **temporal**, precisamos garantir que a série está ordenada e verificar se não há
dias inteiros faltando na sequência — algo que fará diferença mais adiante no Pipeline de ML
(divisão treino/teste por tempo).

In [ ]:
# ordenando os dados pelas datas.
# é necessário refazer os índices
df = df.sort_values("data").reset_index(drop=True)
df.head()

In [ ]:
# Existe algum dia, dentro do período coberto, que não aparece no dataset?
# pd.date_range() -->  cria uma sequência de datas.
todas_as_datas = pd.date_range(
    df["data"].min(),     #Menor data
    df["data"].max(),     #Maior data
    freq="D"              #Frequencia diária (D)
)

# difference() --> compara a sequência completa de datas com as datas presentes na coluna data.
datas_faltando = todas_as_datas.difference(df["data"])

# datas_faltando[:10] --> Seleciona as 10 primeiras datas faltantes
len(datas_faltando), datas_faltando[:10]

Se aparecerem datas faltando, isso **não é para "consertar" agora** — é para **documentar**. Pode
significar falha de coleta em dias específicos, e o grupo decide mais adiante (no Pipeline de ML) como
lidar com essas lacunas.

## 10. Padronizando nomes de colunas

Nosso dataset já está em `snake_case`, mas é comum receber colunas com espaços, acentos ou letras
maiúsculas vindas de outras fontes. O comando para resolver isso é sempre o mesmo:

In [ ]:
df = df.rename(columns={
    "data": "data",
    "cidade": "cidade",
    "temperatura_max_c": "temperatura_max_c",
    "temperatura_min_c": "temperatura_min_c",
    "umidade_pct": "umidade_pct",
    "pressao_hpa": "pressao_hpa",
    "vento_kmh": "vento_kmh",
    "precipitacao_mm": "precipitacao_mm",
})
# Neste dataset os nomes já estavam corretos — o comando acima é o padrão que
# vocês vão reaproveitar quando as colunas vierem com nomes bagunçados nas Trilhas B e C.
df.columns

## 11. Salvando o dataset tratado

Encerramos aqui o Pipeline CD: saímos de um CSV bruto e sujo para um **dataset analítico**, pronto
para a Análise Exploratória de Dados.

Seguindo a arquitetura da proposta do projeto, dado bruto e dado tratado **nunca se misturam**: o
CSV original permanece intocado em `/data/raw`, e o resultado desta limpeza vai para
`/data/processed`.

In [ ]:
df.to_csv("dados_tratados_chuva_trilha_a.csv", index=False)
print("Linhas finais:", df.shape[0])
print("Colunas finais:", list(df.columns))
df.head()

## 12. Checklist — o que fizemos hoje (Pipeline CD)

| # | O que fizemos | Comando(s) principal(is) |
|---|---|---|
| 1 | Ingestão do CSV | `pd.read_csv` |
| 2 | Inspeção inicial (validação de esquema) | `.shape`, `.dtypes`, `.info()`, `.describe()` |
| 3 | Tratamento de valores ausentes essenciais | `.isna()`, `.dropna()` |
| 4 | Remoção de duplicatas | `.duplicated()`, `.drop_duplicates()` |
| 5 | Padronização de texto | `.str.strip()`, `.str.upper()`, `.replace()` |
| 6 | Conversão de tipos (decimais e datas) | `.str.replace()`, `pd.to_numeric()`, `pd.to_datetime()` |
| 7 | Tratamento de outliers por regra de domínio | máscaras booleanas + `.loc` |
| 8 | Ordenação temporal e checagem de lacunas | `.sort_values()`, `pd.date_range()` |
| 9 | Padronização de nomes de colunas | `.rename()` |
| 10 | Exportação do dataset analítico | `.to_csv()` |

### Desafio para levar para o grupo

Pensando na trilha que seu grupo vai escolher (**Qualidade do ar** ou **Produtividade agrícola**):

1. Quais colunas vocês esperam encontrar na API que vão usar?
2. Que tipos de "sujeira" (como as desta aula) vocês imaginam que podem aparecer nos dados reais?
3. Quais seriam as regras de domínio válidas para as variáveis do seu tema (ex.: um índice de
   qualidade do ar tem limite mínimo e máximo? uma safra tem um intervalo de datas plausível?)

Guardem essas respostas — elas vão direto para o Kickoff do grupo (Seção 11.2 da proposta do
projeto).